# Discrete Mathematics for Machine Learning and AI

## Table of Contents
1. [Setup and Imports](#setup)
2. [Combinatorics and Counting](#combinatorics)
3. [Graph Theory and Algorithms](#graph-theory)
4. [Discrete Optimization](#optimization)
5. [Boolean Logic and Circuits](#boolean-logic)
6. [Recurrence Relations and Generating Functions](#recurrence)
7. [Number Theory and Cryptography](#number-theory)
8. [Automata and Formal Languages](#automata)
9. [Discrete Probability](#probability)
10. [Applications in Machine Learning](#ml-applications)

This notebook provides comprehensive coverage of discrete mathematics topics essential for ML/AI, with practical Python implementations and visualizations using NetworkX, NumPy, and Matplotlib.

## 1. Setup and Imports {#setup}

We'll use NetworkX for graph algorithms, NumPy for computations, and Matplotlib for visualizations.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from itertools import combinations, permutations, product
from collections import deque, defaultdict
import math
import random
from scipy.special import comb, factorial
from typing import List, Set, Tuple, Dict
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

# Configure plotting
try:
    plt.style.use('seaborn-v0_8-darkgrid')
except:
    plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

print("✓ All libraries imported successfully")
print(f"NumPy version: {np.__version__}")
print(f"NetworkX version: {nx.__version__}")

## 2. Combinatorics and Counting {#combinatorics}

Combinatorics deals with counting, arranging, and selecting discrete objects. Key concepts include:
- **Permutations**: Ordered arrangements
- **Combinations**: Unordered selections
- **Binomial coefficients**: Pascal's triangle
- **Stirling numbers**: Partitions and cycles
- **Catalan numbers**: Recursive structures

These are fundamental for feature selection, probability, and analyzing discrete structures in ML.

In [ ]:
print("="*80)
print("COMBINATORICS AND COUNTING")
print("="*80)

# 1. Basic Combinatorial Functions
def permutations_count(n, k):
    """P(n,k) = n!/(n-k)!"""
    return int(factorial(n) / factorial(n-k) if k <= n else 0)

def combinations_count(n, k):
    """C(n,k) = n!/(k!(n-k)!)"""
    return int(comb(n, k, exact=True))

# Demonstrate basic counting
n, k = 10, 3
print(f"\n1. Basic Counting (n={n}, k={k}):")
print(f"   Permutations P({n},{k}) = {permutations_count(n, k):,}")
print(f"   Combinations C({n},{k}) = {combinations_count(n, k):,}")

# 2. Stirling Numbers of the Second Kind
def stirling_second(n, k, memo={}):
    """S(n,k): Number of ways to partition n items into k non-empty subsets"""
    if n == 0 and k == 0:
        return 1
    if n == 0 or k == 0 or k > n:
        return 0
    if (n, k) in memo:
        return memo[(n, k)]
    
    result = k * stirling_second(n-1, k, memo) + stirling_second(n-1, k-1, memo)
    memo[(n, k)] = result
    return result

# 3. Bell Numbers
def bell_number(n):
    """B(n): Total number of partitions of n items"""
    return sum(stirling_second(n, k) for k in range(n+1))

# 4. Catalan Numbers
def catalan(n):
    """C_n: Number of valid parenthesizations, binary trees, etc."""
    if n <= 1:
        return 1
    return int(comb(2*n, n, exact=True) // (n + 1))

print(f"\n2. Advanced Combinatorial Numbers:")
for i in range(1, 7):
    print(f"   n={i}: Bell={bell_number(i):4d}, Catalan={catalan(i):4d}, "
          f"Stirling(n,2)={stirling_second(i, 2) if i >= 2 else 0:4d}")

# 5. Visualization: Pascal's Triangle
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Pascal's Triangle
n_rows = 12
pascal = np.zeros((n_rows, n_rows))
for i in range(n_rows):
    for j in range(i+1):
        pascal[i, j] = combinations_count(i, j)

im1 = axes[0].imshow(np.log1p(pascal), cmap='YlOrRd', aspect='auto')
axes[0].set_title("Pascal's Triangle (log scale)", fontsize=12, fontweight='bold')
axes[0].set_xlabel('k')
axes[0].set_ylabel('n')
plt.colorbar(im1, ax=axes[0])

# Binomial Distribution
n_trials = 20
p = 0.5
k_values = np.arange(0, n_trials+1)
probs = [combinations_count(n_trials, k) * (p**k) * ((1-p)**(n_trials-k)) for k in k_values]

axes[1].bar(k_values, probs, color='steelblue', alpha=0.7, edgecolor='black')
axes[1].set_title(f'Binomial Distribution (n={n_trials}, p={p})', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Number of successes')
axes[1].set_ylabel('Probability')
axes[1].grid(True, alpha=0.3)

# Catalan Numbers
cat_n = 15
catalan_nums = [catalan(i) for i in range(cat_n)]
axes[2].semilogy(range(cat_n), catalan_nums, 'o-', color='darkgreen', linewidth=2, markersize=6)
axes[2].set_title('Catalan Numbers (log scale)', fontsize=12, fontweight='bold')
axes[2].set_xlabel('n')
axes[2].set_ylabel('C_n')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 6. Application: Feature Selection Combinations
print(f"\n3. ML Application - Feature Selection:")
total_features = 20
for k in [3, 5, 10]:
    num_combinations = combinations_count(total_features, k)
    print(f"   Selecting {k} features from {total_features}: {num_combinations:,} combinations")

print("\n✓ Combinatorics demonstration complete")

## 3. Graph Theory and Algorithms {#graph-theory}

Graphs model relationships and networks. Key algorithms include:
- **Shortest paths**: Dijkstra, Bellman-Ford, Floyd-Warshall
- **Minimum spanning trees**: Kruskal, Prim
- **Graph coloring**: NP-complete problem with many applications
- **Maximum matching**: Bipartite matching, Hungarian algorithm
- **Network flow**: Max-flow min-cut theorem

Applications: social networks, recommendation systems, routing, scheduling.

In [ ]:
print("="*80)
print("GRAPH THEORY AND ALGORITHMS")
print("="*80)

# 1. Create sample graphs
# Weighted graph for shortest paths and MST
G_weighted = nx.Graph()
edges = [
    ('A', 'B', 4), ('A', 'C', 2), ('B', 'C', 1), ('B', 'D', 5),
    ('C', 'D', 8), ('C', 'E', 10), ('D', 'E', 2), ('D', 'F', 6),
    ('E', 'F', 3)
]
G_weighted.add_weighted_edges_from(edges)

# Bipartite graph for matching
G_bipartite = nx.Graph()
G_bipartite.add_edges_from([
    ('Job1', 'Worker1'), ('Job1', 'Worker2'),
    ('Job2', 'Worker2'), ('Job2', 'Worker3'),
    ('Job3', 'Worker1'), ('Job3', 'Worker3'),
    ('Job4', 'Worker4'), ('Job4', 'Worker2')
])

# Graph for coloring
G_color = nx.petersen_graph()

print("\n1. Graph Properties:")
print(f"   Weighted graph: {G_weighted.number_of_nodes()} nodes, {G_weighted.number_of_edges()} edges")
print(f"   Bipartite graph: {G_bipartite.number_of_nodes()} nodes, {G_bipartite.number_of_edges()} edges")
print(f"   Petersen graph: {G_color.number_of_nodes()} nodes, {G_color.number_of_edges()} edges")

# 2. Shortest Path Algorithms
source, target = 'A', 'F'
shortest_path = nx.shortest_path(G_weighted, source, target, weight='weight')
path_length = nx.shortest_path_length(G_weighted, source, target, weight='weight')

print(f"\n2. Shortest Path ({source} → {target}):")
print(f"   Path: {' → '.join(shortest_path)}")
print(f"   Length: {path_length}")

# All-pairs shortest paths
all_shortest = dict(nx.all_pairs_dijkstra_path_length(G_weighted))

# 3. Minimum Spanning Tree
mst = nx.minimum_spanning_tree(G_weighted)
mst_weight = sum(G_weighted[u][v]['weight'] for u, v in mst.edges())

print(f"\n3. Minimum Spanning Tree:")
print(f"   Edges: {list(mst.edges())}")
print(f"   Total weight: {mst_weight}")

# 4. Graph Coloring
coloring = nx.greedy_color(G_color, strategy='largest_first')
num_colors = max(coloring.values()) + 1

print(f"\n4. Graph Coloring (Petersen Graph):")
print(f"   Chromatic number (approximate): {num_colors}")
print(f"   Color distribution: {np.bincount(list(coloring.values()))}")

# 5. Maximum Matching
matching = nx.max_weight_matching(G_bipartite)

print(f"\n5. Maximum Matching (Bipartite):")
print(f"   Matching size: {len(matching)}")
print(f"   Matched pairs: {sorted(matching)}")

# 6. Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Shortest path visualization
pos1 = nx.spring_layout(G_weighted, seed=42)
nx.draw_networkx_nodes(G_weighted, pos1, node_color='lightblue', 
                       node_size=700, ax=axes[0, 0])
nx.draw_networkx_labels(G_weighted, pos1, font_size=10, ax=axes[0, 0])

# Draw all edges in gray
nx.draw_networkx_edges(G_weighted, pos1, edge_color='gray', width=1, ax=axes[0, 0])

# Highlight shortest path
path_edges = list(zip(shortest_path[:-1], shortest_path[1:]))
nx.draw_networkx_edges(G_weighted, pos1, edgelist=path_edges, 
                      edge_color='red', width=3, ax=axes[0, 0])

# Edge labels
edge_labels = nx.get_edge_attributes(G_weighted, 'weight')
nx.draw_networkx_edge_labels(G_weighted, pos1, edge_labels, font_size=8, ax=axes[0, 0])
axes[0, 0].set_title(f'Shortest Path: {source} → {target}', fontsize=12, fontweight='bold')
axes[0, 0].axis('off')

# MST visualization
pos2 = pos1  # Use same layout
nx.draw_networkx_nodes(G_weighted, pos2, node_color='lightgreen', 
                       node_size=700, ax=axes[0, 1])
nx.draw_networkx_labels(G_weighted, pos2, font_size=10, ax=axes[0, 1])
nx.draw_networkx_edges(G_weighted, pos2, edge_color='lightgray', 
                      width=1, style='dashed', ax=axes[0, 1])
nx.draw_networkx_edges(mst, pos2, edge_color='darkgreen', width=3, ax=axes[0, 1])

mst_edge_labels = {(u, v): G_weighted[u][v]['weight'] for u, v in mst.edges()}
nx.draw_networkx_edge_labels(mst, pos2, mst_edge_labels, font_size=8, ax=axes[0, 1])
axes[0, 1].set_title(f'Minimum Spanning Tree (weight={mst_weight})', 
                     fontsize=12, fontweight='bold')
axes[0, 1].axis('off')

# Graph coloring
pos3 = nx.shell_layout(G_color)
node_colors = [coloring[node] for node in G_color.nodes()]
nx.draw(G_color, pos3, node_color=node_colors, cmap=plt.cm.rainbow,
        node_size=500, with_labels=True, font_size=8, 
        edge_color='gray', ax=axes[1, 0])
axes[1, 0].set_title(f'Graph Coloring ({num_colors} colors)', 
                     fontsize=12, fontweight='bold')

# Bipartite matching
pos4 = nx.bipartite_layout(G_bipartite, ['Job1', 'Job2', 'Job3', 'Job4'])
node_colors_bip = ['lightcoral' if 'Job' in node else 'lightskyblue' 
                   for node in G_bipartite.nodes()]
nx.draw_networkx_nodes(G_bipartite, pos4, node_color=node_colors_bip, 
                       node_size=800, ax=axes[1, 1])
nx.draw_networkx_labels(G_bipartite, pos4, font_size=9, ax=axes[1, 1])
nx.draw_networkx_edges(G_bipartite, pos4, edge_color='lightgray', 
                      width=1, ax=axes[1, 1])
nx.draw_networkx_edges(G_bipartite, pos4, edgelist=matching, 
                      edge_color='red', width=3, ax=axes[1, 1])
axes[1, 1].set_title('Maximum Bipartite Matching', fontsize=12, fontweight='bold')
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

print("\n✓ Graph theory demonstration complete")

## 4. Discrete Optimization {#optimization}

Discrete optimization problems involve finding the best solution from a finite set. Classic problems include:
- **Knapsack Problem**: Resource allocation with constraints
- **Traveling Salesman**: Finding shortest tour
- **Integer Programming**: Optimization with discrete variables
- **Set Cover**: Covering elements with minimum sets

These are NP-hard problems with applications in scheduling, resource allocation, and ML model selection.

In [ ]:
print("="*80)
print("DISCRETE OPTIMIZATION")
print("="*80)

# 1. Knapsack Problem (0/1 Knapsack using Dynamic Programming)
def knapsack_01(weights, values, capacity):
    """
    Solve 0/1 knapsack problem using dynamic programming.
    Returns: (max_value, selected_items)
    """
    n = len(weights)
    # dp[i][w] = maximum value using first i items with capacity w
    dp = [[0 for _ in range(capacity + 1)] for _ in range(n + 1)]
    
    # Fill the DP table
    for i in range(1, n + 1):
        for w in range(capacity + 1):
            # Don't include item i-1
            dp[i][w] = dp[i-1][w]
            
            # Include item i-1 if it fits
            if weights[i-1] <= w:
                dp[i][w] = max(dp[i][w], 
                              dp[i-1][w - weights[i-1]] + values[i-1])
    
    # Backtrack to find selected items
    selected = []
    w = capacity
    for i in range(n, 0, -1):
        if dp[i][w] != dp[i-1][w]:
            selected.append(i-1)
            w -= weights[i-1]
    
    return dp[n][capacity], selected[::-1]

# Example problem
weights = [2, 3, 4, 5, 6]
values = [3, 4, 5, 8, 10]
capacity = 10

max_value, selected_items = knapsack_01(weights, values, capacity)

print("\n1. Knapsack Problem (0/1):")
print(f"   Weights: {weights}")
print(f"   Values: {values}")
print(f"   Capacity: {capacity}")
print(f"   Maximum value: {max_value}")
print(f"   Selected items (indices): {selected_items}")
print(f"   Total weight: {sum(weights[i] for i in selected_items)}")

# 2. Traveling Salesman Problem (Approximate solution using greedy)
def tsp_nearest_neighbor(distance_matrix, start=0):
    """
    Solve TSP using nearest neighbor heuristic.
    Returns: (tour, total_distance)
    """
    n = len(distance_matrix)
    unvisited = set(range(n))
    tour = [start]
    unvisited.remove(start)
    
    current = start
    total_distance = 0
    
    while unvisited:
        # Find nearest unvisited city
        nearest = min(unvisited, key=lambda x: distance_matrix[current][x])
        total_distance += distance_matrix[current][nearest]
        tour.append(nearest)
        current = nearest
        unvisited.remove(nearest)
    
    # Return to start
    total_distance += distance_matrix[current][start]
    tour.append(start)
    
    return tour, total_distance

# Create random distance matrix
n_cities = 8
np.random.seed(42)
# Generate random points
city_coords = np.random.rand(n_cities, 2) * 100
# Compute distance matrix
dist_matrix = np.zeros((n_cities, n_cities))
for i in range(n_cities):
    for j in range(n_cities):
        dist_matrix[i][j] = np.linalg.norm(city_coords[i] - city_coords[j])

tour, tour_distance = tsp_nearest_neighbor(dist_matrix)

print(f"\n2. Traveling Salesman Problem (Nearest Neighbor):")
print(f"   Number of cities: {n_cities}")
print(f"   Tour: {tour}")
print(f"   Total distance: {tour_distance:.2f}")

# 3. Set Cover Problem (Greedy approximation)
def set_cover_greedy(universe, subsets):
    """
    Greedy approximation for set cover problem.
    Returns: list of selected subset indices
    """
    uncovered = set(universe)
    selected = []
    
    while uncovered:
        # Find subset covering most uncovered elements
        best_idx = max(range(len(subsets)),
                      key=lambda i: len(subsets[i] & uncovered))
        
        if len(subsets[best_idx] & uncovered) == 0:
            break
            
        selected.append(best_idx)
        uncovered -= subsets[best_idx]
    
    return selected

# Example set cover problem
universe = set(range(1, 11))  # {1, 2, ..., 10}
subsets = [
    {1, 2, 3, 4},
    {3, 4, 5, 6},
    {6, 7, 8},
    {8, 9, 10},
    {1, 5, 9},
    {2, 6, 10}
]

cover = set_cover_greedy(universe, subsets)

print(f"\n3. Set Cover Problem (Greedy):")
print(f"   Universe: {sorted(universe)}")
print(f"   Number of subsets: {len(subsets)}")
print(f"   Selected subsets: {cover}")
print(f"   Coverage: {sorted(set.union(*[subsets[i] for i in cover]))}")

# 4. Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Knapsack visualization
x_pos = np.arange(len(weights))
colors = ['green' if i in selected_items else 'gray' for i in range(len(weights))]
axes[0].bar(x_pos, values, color=colors, alpha=0.7, edgecolor='black')
axes[0].set_xlabel('Item Index')
axes[0].set_ylabel('Value')
axes[0].set_title('Knapsack Solution (green = selected)', fontweight='bold')
axes[0].set_xticks(x_pos)
axes[0].grid(True, alpha=0.3, axis='y')

# Add weight labels
for i, (w, v) in enumerate(zip(weights, values)):
    axes[0].text(i, v + 0.3, f'w={w}', ha='center', fontsize=9)

# TSP visualization
axes[1].scatter(city_coords[:, 0], city_coords[:, 1], c='red', s=200, zorder=3)
for i, (x, y) in enumerate(city_coords):
    axes[1].annotate(str(i), (x, y), ha='center', va='center', 
                    color='white', fontweight='bold', fontsize=10)

# Draw tour
for i in range(len(tour) - 1):
    start, end = tour[i], tour[i+1]
    axes[1].plot([city_coords[start, 0], city_coords[end, 0]],
                [city_coords[start, 1], city_coords[end, 1]],
                'b-', linewidth=2, alpha=0.6)
    
    # Add arrow
    mid_x = (city_coords[start, 0] + city_coords[end, 0]) / 2
    mid_y = (city_coords[start, 1] + city_coords[end, 1]) / 2
    dx = city_coords[end, 0] - city_coords[start, 0]
    dy = city_coords[end, 1] - city_coords[start, 1]
    axes[1].arrow(mid_x - dx*0.1, mid_y - dy*0.1, dx*0.1, dy*0.1,
                 head_width=3, head_length=2, fc='blue', ec='blue', alpha=0.6)

axes[1].set_xlabel('X coordinate')
axes[1].set_ylabel('Y coordinate')
axes[1].set_title(f'TSP Tour (distance={tour_distance:.1f})', fontweight='bold')
axes[1].grid(True, alpha=0.3)

# Set cover visualization
# Create a binary matrix showing which elements are in which subsets
cover_matrix = np.zeros((len(subsets), len(universe)))
for i, subset in enumerate(subsets):
    for elem in subset:
        cover_matrix[i, elem-1] = 1

# Highlight selected subsets
row_colors = ['green' if i in cover else 'white' for i in range(len(subsets))]
im = axes[2].imshow(cover_matrix, cmap='Blues', aspect='auto')

# Add grid
for i in range(len(subsets)):
    for j in range(len(universe)):
        if cover_matrix[i, j] == 1:
            axes[2].text(j, i, '✓', ha='center', va='center', 
                        color='darkblue', fontweight='bold')
        # Highlight selected rows
        if i in cover:
            axes[2].add_patch(plt.Rectangle((j-0.5, i-0.5), 1, 1, 
                                            fill=False, edgecolor='red', linewidth=2))

axes[2].set_xlabel('Elements')
axes[2].set_ylabel('Subsets')
axes[2].set_title('Set Cover (red boxes = selected)', fontweight='bold')
axes[2].set_xticks(range(len(universe)))
axes[2].set_xticklabels(range(1, len(universe)+1))
axes[2].set_yticks(range(len(subsets)))

plt.tight_layout()
plt.show()

print("\n✓ Discrete optimization demonstration complete")

## 5. Boolean Logic and Circuits {#boolean-logic}

Boolean algebra deals with binary values and logical operations. Key concepts:
- **Logical operations**: AND, OR, NOT, XOR, NAND, NOR
- **Boolean functions**: Truth tables, canonical forms
- **Circuit minimization**: Karnaugh maps, Quine-McCluskey
- **SAT problems**: Boolean satisfiability (NP-complete)

Applications: digital circuits, logic programming, constraint satisfaction in AI.

In [ ]:
print("="*80)print("BOOLEAN LOGIC AND CIRCUITS")print("="*80)# 1. Basic Boolean operationsdef AND(a, b): return a and bdef OR(a, b): return a or b  def NOT(a): return not adef XOR(a, b): return (a and not b) or (not a and b)def NAND(a, b): return not (a and b)print("\n1. Basic Boolean Operations:")print("   A B | AND OR XOR NAND")print("   ----|----------------")for a in [0, 1]:    for b in [0, 1]:        print(f"   {a} {b} |  {int(AND(a,b))}  {int(OR(a,b))}   {int(XOR(a,b))}   {int(NAND(a,b))}")# 2. Boolean function analysisclass BooleanFunction:    def __init__(self, n_vars, truth_table):        self.n_vars = n_vars        self.truth_table = truth_table        def evaluate(self, *inputs):        index = sum(bit << i for i, bit in enumerate(reversed(inputs)))        return self.truth_table[index]        def count_ones(self):        return sum(self.truth_table)# Create some 3-variable functionsmajority = BooleanFunction(3, [0, 0, 0, 1, 0, 1, 1, 1])  # At least 2 inputs are 1parity = BooleanFunction(3, [0, 1, 1, 0, 1, 0, 0, 1])    # XOR of all inputsprint("\n2. 3-Variable Boolean Functions:")print(f"   Majority function has {majority.count_ones()} ones")print(f"   Parity function has {parity.count_ones()} ones")# 3. SAT Solver (simplified)class SimpleSAT:    def __init__(self):        self.clauses = []        def add_clause(self, literals):        self.clauses.append(literals)        def check_assignment(self, assignment):        for clause in self.clauses:            satisfied = any((lit > 0 and assignment.get(lit)) or                            (lit < 0 and not assignment.get(-lit))                           for lit in clause)            if not satisfied:                return False        return True# Example: (x1 OR x2) AND (NOT x1 OR x3) sat = SimpleSAT()sat.add_clause([1, 2])       # x1 OR x2sat.add_clause([-1, 3])      # NOT x1 OR x3# Test assignmentassignment = {1: True, 2: False, 3: True}print(f"\n3. SAT Problem:")print(f"   Clauses: (x1 OR x2) AND (NOT x1 OR x3)")print(f"   Assignment {assignment}: {sat.check_assignment(assignment)}")# 4. Visualizationfig, axes = plt.subplots(1, 3, figsize=(15, 5))# Truth table heatmapfunctions_3var = {    'Majority': [0, 0, 0, 1, 0, 1, 1, 1],    'Parity': [0, 1, 1, 0, 1, 0, 0, 1],    'AND': [0, 0, 0, 0, 0, 0, 0, 1],    'OR': [0, 1, 1, 1, 1, 1, 1, 1]}func_matrix = np.array([functions_3var[name] for name in functions_3var.keys()])im = axes[0].imshow(func_matrix, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)axes[0].set_yticks(range(len(functions_3var)))axes[0].set_yticklabels(functions_3var.keys())axes[0].set_xticks(range(8))axes[0].set_xticklabels([f'{i:03b}' for i in range(8)])axes[0].set_xlabel('Input (binary)')axes[0].set_title('3-Variable Boolean Functions', fontweight='bold')plt.colorbar(im, ax=axes[0])# Boolean function complexity distributionn_vars = 4sample_size = 10000ones_counts = []for _ in range(sample_size):    truth_table = np.random.randint(0, 2, 2**n_vars)    ones_counts.append(sum(truth_table))axes[1].hist(ones_counts, bins=range(2**n_vars + 2),            color='steelblue', alpha=0.7, edgecolor='black')axes[1].axvline(2**(n_vars-1), color='red', linestyle='--', linewidth=2, label='Expected mean')axes[1].set_xlabel('Number of 1s in truth table')axes[1].set_ylabel('Frequency')axes[1].set_title(f'Boolean Function Complexity ({n_vars} vars)', fontweight='bold')axes[1].legend()axes[1].grid(True, alpha=0.3, axis='y')# Logic circuitcircuit = nx.DiGraph()circuit.add_edges_from([    ('x1', 'AND'), ('x2', 'AND'),    ('x2', 'OR'), ('x3', 'OR'),    ('AND', 'XOR'), ('OR', 'XOR'),    ('XOR', 'out')])pos = {    'x1': (0, 2), 'x2': (0, 1), 'x3': (0, 0),    'AND': (1, 1.5), 'OR': (1, 0.5),    'XOR': (2, 1), 'out': (3, 1)}node_colors = ['lightblue' if n.startswith('x') else 'lightcoral' if n=='out' else 'lightgreen'                for n in circuit.nodes()]nx.draw(circuit, pos, with_labels=True, node_color=node_colors,       node_size=1200, font_size=10, font_weight='bold',       arrows=True, arrowsize=15, edge_color='gray', ax=axes[2])axes[2].set_title('Boolean Circuit Example', fontweight='bold')plt.tight_layout()plt.show()print("\n✓ Boolean logic demonstration complete")

## 6. Recurrence Relations {#recurrence}Recurrence relations define sequences recursively:- **Fibonacci**: F(n) = F(n-1) + F(n-2)- **Master Theorem**: For analyzing divide-and-conquer algorithms- **Generating functions**: Power series representationsApplications: Algorithm complexity analysis, dynamic programming.

In [ ]:
print("="*80)print("RECURRENCE RELATIONS")print("="*80)# 1. Classic sequencesdef fibonacci(n, memo={}):    if n in memo: return memo[n]    if n <= 1: return n    result = fibonacci(n-1, memo) + fibonacci(n-2, memo)    memo[n] = result    return resultdef tribonacci(n, memo={}):    if n in memo: return memo[n]    if n == 0: return 0    if n <= 2: return 1    result = tribonacci(n-1, memo) + tribonacci(n-2, memo) + tribonacci(n-3, memo)    memo[n] = result    return resultprint("\n1. Classic Sequences:")print("   n:  ", " ".join(f"{i:4d}" for i in range(10)))print("   Fib:", " ".join(f"{fibonacci(i):4d}" for i in range(10)))print("   Tri:", " ".join(f"{tribonacci(i):4d}" for i in range(10)))# 2. Master Theoremdef master_theorem(a, b, k):    """Analyze T(n) = a*T(n/b) + O(n^k)"""    c = np.log(a) / np.log(b)    if c > k:        return f"Θ(n^{c:.2f})"    elif c == k:        return f"Θ(n^{k} log n)"    else:        return f"Θ(n^{k})"print("\n2. Master Theorem Examples:")algorithms = [    ("Binary Search", 1, 2, 0),    ("Merge Sort", 2, 2, 1),    ("Strassen", 7, 2, 2)]for name, a, b, k in algorithms:    print(f"   {name:20s}: T(n)={a}T(n/{b})+O(n^{k}) → {master_theorem(a, b, k)}")# 3. Visualizationfig, axes = plt.subplots(1, 3, figsize=(15, 5))# Sequence growthn_terms = 20fib_seq = [fibonacci(i) for i in range(n_terms)]tri_seq = [tribonacci(i) for i in range(n_terms)]axes[0].semilogy(range(n_terms), fib_seq, 'o-', label='Fibonacci', linewidth=2, markersize=5)axes[0].semilogy(range(n_terms), tri_seq, 's-', label='Tribonacci', linewidth=2, markersize=5)axes[0].set_xlabel('n')axes[0].set_ylabel('Value (log scale)')axes[0].set_title('Recurrence Sequences', fontweight='bold')axes[0].legend()axes[0].grid(True, alpha=0.3)# Golden ratio convergenceratios = [fib_seq[i+1]/fib_seq[i] for i in range(len(fib_seq)-1) if fib_seq[i] > 0]golden = (1 + np.sqrt(5)) / 2axes[1].plot(range(len(ratios)), ratios, 'bo-', linewidth=2, label='F(n+1)/F(n)')axes[1].axhline(golden, color='red', linestyle='--', linewidth=2, label=f'φ = {golden:.6f}')axes[1].set_xlabel('n')axes[1].set_ylabel('Ratio')axes[1].set_title('Convergence to Golden Ratio', fontweight='bold')axes[1].legend()axes[1].grid(True, alpha=0.3)# Complexity classesn_values = np.logspace(1, 4, 100)complexities = {    'O(log n)': np.log(n_values),    'O(n)': n_values,    'O(n log n)': n_values * np.log(n_values),    'O(n²)': n_values ** 2}for name, values in complexities.items():    axes[2].loglog(n_values, values, linewidth=2, label=name)axes[2].set_xlabel('n (log scale)')axes[2].set_ylabel('Operations (log scale)')axes[2].set_title('Algorithm Complexity Classes', fontweight='bold')axes[2].legend()axes[2].grid(True, alpha=0.3, which='both')plt.tight_layout()plt.show()print("\n✓ Recurrence relations demonstration complete")

## 7. Discrete Probability {#probability}Discrete probability deals with countable sample spaces:- **Random variables**: Discrete distributions- **Markov chains**: State transitions- **Random graphs**: Erdős-Rényi modelApplications: Probabilistic ML models, randomized algorithms, network analysis.

In [ ]:
print("="*80)print("DISCRETE PROBABILITY")print("="*80)# 1. Discrete distributionsprint("\n1. Discrete Probability Distributions:")# Binomialfrom scipy.stats import binomn_trials = 20p = 0.6k_vals = np.arange(0, n_trials + 1)binomial_probs = binom.pmf(k_vals, n_trials, p)print(f"   Binomial(n={n_trials}, p={p}):")print(f"   Mean: {n_trials * p:.2f}, Variance: {n_trials * p * (1-p):.2f}")# Poissonfrom scipy.stats import poissonlambda_param = 5poisson_probs = poisson.pmf(k_vals, lambda_param)print(f"   Poisson(λ={lambda_param}):")print(f"   Mean: {lambda_param:.2f}, Variance: {lambda_param:.2f}")# 2. Markov Chainclass MarkovChain:    def __init__(self, transition_matrix):        self.P = np.array(transition_matrix)        self.n_states = len(self.P)    def simulate(self, initial_state, n_steps):        states = [initial_state]        current = initial_state        for _ in range(n_steps):            current = np.random.choice(self.n_states, p=self.P[current])            states.append(current)        return states    def stationary_distribution(self):        # Solve πP = π        eigenvalues, eigenvectors = np.linalg.eig(self.P.T)        idx = np.argmax(np.abs(eigenvalues - 1.0) < 1e-10)        stationary = np.real(eigenvectors[:, idx])        return stationary / stationary.sum()# Example: Weather modelP = np.array([    [0.7, 0.3],    # Sunny -> Sunny, Sunny -> Rainy    [0.4, 0.6]     # Rainy -> Sunny, Rainy -> Rainy])mc = MarkovChain(P)stationary = mc.stationary_distribution()print(f"\n2. Markov Chain (Weather Model):")print(f"   Transition matrix:\n{P}")print(f"   Stationary distribution: Sunny={stationary[0]:.3f}, Rainy={stationary[1]:.3f}")# Simulatetrajectory = mc.simulate(0, 100)print(f"   After 100 steps: {sum(1 for s in trajectory if s==0)/len(trajectory):.3f} sunny")# 3. Random Graphsdef erdos_renyi(n, p):    """Generate Erdős-Rényi random graph"""    G = nx.Graph()    G.add_nodes_from(range(n))    for i in range(n):        for j in range(i+1, n):            if np.random.random() < p:                G.add_edge(i, j)    return GG_random = erdos_renyi(20, 0.2)print(f"\n3. Random Graph G(n=20, p=0.2):")print(f"   Nodes: {G_random.number_of_nodes()}, Edges: {G_random.number_of_edges()}")print(f"   Expected edges: {20*19/2 * 0.2:.1f}")print(f"   Connected: {nx.is_connected(G_random)}")# 4. Visualizationfig, axes = plt.subplots(2, 2, figsize=(12, 10))# Distributionsaxes[0, 0].bar(k_vals, binomial_probs, alpha=0.7, label=f'Binomial({n_trials}, {p})', color='steelblue')axes[0, 0].set_xlabel('k')axes[0, 0].set_ylabel('P(X=k)')axes[0, 0].set_title('Binomial Distribution', fontweight='bold')axes[0, 0].legend()axes[0, 0].grid(True, alpha=0.3, axis='y')axes[0, 1].bar(k_vals, poisson_probs, alpha=0.7, label=f'Poisson({lambda_param})', color='coral')axes[0, 1].set_xlabel('k')axes[0, 1].set_ylabel('P(X=k)')axes[0, 1].set_title('Poisson Distribution', fontweight='bold')axes[0, 1].legend()axes[0, 1].grid(True, alpha=0.3, axis='y')# Markov chain trajectoryaxes[1, 0].plot(trajectory, 'o-', markersize=3, linewidth=1, alpha=0.7)axes[1, 0].set_xlabel('Time step')axes[1, 0].set_ylabel('State')axes[1, 0].set_yticks([0, 1])axes[1, 0].set_yticklabels(['Sunny', 'Rainy'])axes[1, 0].set_title('Markov Chain Simulation', fontweight='bold')axes[1, 0].grid(True, alpha=0.3)# Random graphpos = nx.spring_layout(G_random, seed=42)nx.draw(G_random, pos, node_color='lightblue', node_size=300,       with_labels=True, font_size=8, edge_color='gray', ax=axes[1, 1])axes[1, 1].set_title('Random Graph G(20, 0.2)', fontweight='bold')plt.tight_layout()plt.show()print("\n✓ Discrete probability demonstration complete")

## 8. Applications in Machine Learning {#ml-applications}Discrete mathematics applications in ML/AI:- **Feature selection**: Combinatorial optimization- **Network analysis**: Graph algorithms for social networks- **Discrete latent variables**: Hidden Markov Models, mixture models- **Constraint satisfaction**: Logic in knowledge representation- **Complexity theory**: Understanding computational limitsThis section demonstrates practical ML applications.

In [ ]:
print("="*80)print("MACHINE LEARNING APPLICATIONS")print("="*80)# 1. Feature Selection (Combinatorial)print("\n1. Feature Selection:")# Simulate datasetnp.random.seed(42)n_samples = 100n_features = 10X = np.random.randn(n_samples, n_features)# Make features 0, 2, 5 relevanttrue_coef = np.zeros(n_features)true_coef[[0, 2, 5]] = [2.5, -1.5, 3.0]y = X @ true_coef + np.random.randn(n_samples) * 0.5# Greedy feature selectionfrom sklearn.linear_model import LinearRegressiondef greedy_feature_selection(X, y, k):    """Select k features greedily"""    selected = []    remaining = list(range(X.shape[1]))    for _ in range(k):        best_score = -np.inf        best_feature = None        for feat in remaining:            features = selected + [feat]            model = LinearRegression()            model.fit(X[:, features], y)            score = model.score(X[:, features], y)            if score > best_score:                best_score = score                best_feature = feat        selected.append(best_feature)        remaining.remove(best_feature)    return selectedk_features = 5selected_features = greedy_feature_selection(X, y, k_features)print(f"   Total features: {n_features}")print(f"   Selected {k_features} features: {selected_features}")print(f"   True important features: [0, 2, 5]")print(f"   Overlap: {set(selected_features) & {0, 2, 5}}")# 2. Community Detection (Graph Clustering)print("\n2. Community Detection in Networks:")# Create graph with communitiesG_comm = nx.karate_club_graph()# Detect communities using greedy modularityfrom networkx.algorithms import communitycommunities = community.greedy_modularity_communities(G_comm)print(f"   Nodes: {G_comm.number_of_nodes()}, Edges: {G_comm.number_of_edges()}")print(f"   Detected {len(communities)} communities")for i, comm in enumerate(communities):    print(f"   Community {i}: {len(comm)} members")# 3. Discrete Latent Variables (Mixture Model)print("\n3. Discrete Latent Variables (Gaussian Mixture):")from sklearn.mixture import GaussianMixture# Generate data from 3 Gaussiansn_components = 3X_mix = np.vstack([    np.random.randn(50, 2) + [0, 0],    np.random.randn(50, 2) + [5, 5],    np.random.randn(50, 2) + [5, 0]])gmm = GaussianMixture(n_components=n_components, random_state=42)labels = gmm.fit_predict(X_mix)print(f"   Fitted {n_components}-component Gaussian Mixture")print(f"   Component weights: {gmm.weights_}")print(f"   Converged: {gmm.converged_}")# 4. Constraint Satisfaction (Example)print("\n4. Constraint Satisfaction (Example):")# Simple constraint propagation exampledef is_valid_assignment(variables, constraints):    """Check if variable assignment satisfies constraints"""    for var1, var2, relation in constraints:        if var1 in variables and var2 in variables:            if not relation(variables[var1], variables[var2]):                return False    return True# Example: x1 < x2, x2 < x3, x1 + x2 < x3variables = {'x1': 1, 'x2': 2, 'x3': 3}constraints = [    ('x1', 'x2', lambda a, b: a < b),    ('x2', 'x3', lambda a, b: a < b),    ('x1', 'x3', lambda a, b: a < b)]print(f"   Variables: {variables}")print(f"   Satisfies constraints: {is_valid_assignment(variables, constraints)}")# 5. Visualizationfig = plt.figure(figsize=(15, 10))gs = fig.add_gridspec(2, 3)# Feature importanceax1 = fig.add_subplot(gs[0, 0])colors = ['green' if i in selected_features else 'gray' for i in range(n_features)]ax1.bar(range(n_features), np.abs(true_coef), color=colors, alpha=0.7, edgecolor='black')ax1.set_xlabel('Feature index')ax1.set_ylabel('True coefficient magnitude')ax1.set_title('Feature Selection (green=selected)', fontweight='bold')ax1.grid(True, alpha=0.3, axis='y')# Community detectionax2 = fig.add_subplot(gs[0, 1])pos = nx.spring_layout(G_comm, seed=42)color_map = []for node in G_comm:    for i, comm in enumerate(communities):        if node in comm:            color_map.append(i)            breaknx.draw(G_comm, pos, node_color=color_map, cmap=plt.cm.rainbow,       node_size=200, with_labels=True, font_size=8, ax=ax2)ax2.set_title('Community Detection (Karate Club)', fontweight='bold')# Clustering (GMM)ax3 = fig.add_subplot(gs[0, 2])scatter = ax3.scatter(X_mix[:, 0], X_mix[:, 1], c=labels, cmap='viridis',                     alpha=0.6, edgecolors='black', linewidth=0.5)ax3.scatter(gmm.means_[:, 0], gmm.means_[:, 1], c='red', s=200,           marker='X', edgecolors='black', linewidth=2, label='Centroids')ax3.set_xlabel('Feature 1')ax3.set_ylabel('Feature 2')ax3.set_title('Gaussian Mixture Model Clustering', fontweight='bold')ax3.legend()ax3.grid(True, alpha=0.3)# Feature selection convergenceax4 = fig.add_subplot(gs[1, :])# Show how R² improves with more featuresr2_scores = []for k in range(1, n_features + 1):    sel_feats = greedy_feature_selection(X, y, k)    model = LinearRegression()    model.fit(X[:, sel_feats], y)    r2_scores.append(model.score(X[:, sel_feats], y))ax4.plot(range(1, n_features + 1), r2_scores, 'bo-', linewidth=2, markersize=8)ax4.axhline(1.0, color='red', linestyle='--', alpha=0.5, label='Perfect fit')ax4.set_xlabel('Number of features selected')ax4.set_ylabel('R² Score')ax4.set_title('Greedy Feature Selection Performance', fontweight='bold')ax4.legend()ax4.grid(True, alpha=0.3)plt.tight_layout()plt.show()print("\n✓ ML applications demonstration complete")print("\n" + "="*80)print("All discrete mathematics topics completed!")print("="*80)

## SummaryThis notebook covered key discrete mathematics topics for ML/AI:1. **Combinatorics**: Counting, permutations, combinations, binomial coefficients2. **Graph Theory**: Shortest paths, MST, graph coloring, matching algorithms3. **Discrete Optimization**: Knapsack, TSP, set cover problems4. **Boolean Logic**: Truth tables, SAT problems, circuit design5. **Recurrence Relations**: Fibonacci, complexity analysis, Master Theorem6. **Discrete Probability**: Distributions, Markov chains, random graphs7. **ML Applications**: Feature selection, community detection, clusteringThese mathematical foundations are essential for:- Algorithm design and analysis- Complexity theory and computational limits- Network analysis and graph algorithms- Constraint satisfaction and logic programming- Probabilistic models and randomized algorithms- Combinatorial optimization in MLThe implementations demonstrate practical applications using NetworkX, NumPy, and scikit-learn.